## Definimos el entorno
al definir el entorno en este caso el tablero incluimos los Sensores del Agente
Los sensores son cómo el agente percibe el entorno.

El agente percibe el estado actual del tablero 4×5:

Dónde está su ficha (X = 1)
Dónde está la ficha del oponente (O = -1)
Dónde hay posiciones vacías (0)

In [ ]:
import numpy as np


class Board():
    def __init__(self):
        self.state = np.zeros((4,5))

    def valid_moves(self):
        return [(i, j) for i in range(4) for j in range(5) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] == 0:
            self.state[row, col] = symbol
        else:
            raise ValueError ("movimiento ilegal, posicion ya marcada !")

    def is_game_over(self):

        # comprobar filas y columnas
        if (self.state.sum(axis=0) == 3).sum() >= 1 or (self.state.sum(axis=1) == 3).sum() >= 1:
            return 1
        if (self.state.sum(axis=0) == -3).sum() >= 1 or (self.state.sum(axis=1) == -3).sum() >= 1:
            return -1
        # comprobar diagonales
        # DIAGONALES \ 
        for i in range(2):
            for j in range(3):
                d_sum = self.state[i, j] + self.state[i+1, j+1] + self.state[i+2, j+2]
                if d_sum == 3:
                    return 1
                if d_sum == -3:
                    return -1
    
        # DIAGONALES /
        for i in range(2):
            for j in range(2, 5):
                d_sum = self.state[i, j] + self.state[i+1, j-1] + self.state[i+2, j-2]
                if d_sum == 3:
                    return 1
                if d_sum == -3:
                    return -1
        # empate
        if len(self.valid_moves()) == 0:
            return 0
        # seguir jugando
        return None

    def reset(self):
        self.state = np.zeros((4,5))

In [26]:
from tqdm import tqdm

class Game():
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins


    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0: # empate
            for player in self.players:
                player.reward(0.5)
        else: # le damos 1 recompensa al jugador que gana
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(0)

## Acciones del Agente
Las acciones son lo que el agente puede hacer.
El agente puede ejecutar 20 acciones posibles (una por cada casilla del tablero 4×5):
Puede Colocar su ficha (X) en cualquier posición vacía
Total: 4 filas × 5 columnas = 20 posiciones
Determinamos acciones validas e inalidas
Acción válida: Poner X en una posición vacía → Se ejecuta
Acción inválida: Intentar poner X donde ya hay ficha → Error (raramente ocurre con buen entrenamiento)


In [27]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {} # tabla con pares estado -> valor
        self.alpha = alpha         # learning rate
        self.positions = []       # guardamos todas las posiciones de la partida
        self.prob_exp = prob_exp   # probabilidad de explorar

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        # exploracion
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            # vamos a una posición aleatoria
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        # explotacion
        # vamos a la posición con más valor
        max_value = -1000
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(4*5))
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(4*5)))


    def reward(self, reward):
        # al final de la partida (cuando recibimos la recompensa)
        # iteramos por tods los estados actualizando su valor en la tabla
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

## Exploración vs Explotación (ε-Greedy)
Este es el balance más importante del Aprendizaje por Refuerzo.


Explotar : usar movimientos que sé que son buenos
Explorar: Probar movimientos nuevos para descubrir movimientos con más recompensa

En este caso al agente 1 le pusimos un 60% de probabilidad de exploración y al otro lo dejamos con el por defecto que es 50%

In [ ]:
agent1 = Agent(prob_exp=0.6)
agent2 = Agent()

game = Game(agent1, agent2)

game.selfplay(30000)

  0%|          | 0/30000 [00:00<?, ?it/s]

100%|██████████| 30000/30000 [03:34<00:00, 140.03it/s]


[18276, 11645]

## Objetivo
El objetivo del agente Es la función de valor, una tabla donde guarda los valores de recompensas conseguidas para que el agente aprenda que camino le llevara a una recompensa máxima.

In [30]:
import pandas as pd

funcion_de_valor = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)
tabla = pd.DataFrame({'estado': [x[0] for x in funcion_de_valor], 'valor': [x[1] for x in funcion_de_valor]})

tabla

,estado,valor
0,[ 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...,1.0
1,[ 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 0. ...,1.0
2,[ 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. ...,1.0
3,[ 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. ...,1.0
4,[ 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. ...,1.0
...,...,...
158096,[ 0. 0. 0. 0. 1. 0. -1. 0. 0. 0. 0. ...,0.0
158097,[ 0. 0. 0. 0. 1. 0. -1. 0. 0. 0. 0. ...,0.0
158098,[ 0. 0. 0. 0. 0. 0. -1. 0. 0. 0. 0. ...,0.0
158099,[ 0. 0. 0. 0. 0. 0. -1. 0. 0. 0. 0. ...,0.0


Guardamos la función de valor en un archivo para que después podamos entrenar a un agente con esa función.

In [31]:
import pickle

with open('agente4x5.pickle', 'wb') as handle:
    pickle.dump(agent1.value_function, handle, protocol=pickle.HIGHEST_PROTOCOL)